#### CHECK THE FILES FOR PAYMENTS

In [0]:
%fs ls 'abfss://gizmo@gizmoboxdevadls.dfs.core.windows.net/landing/external/payments'

#### READ `PAYMENTS` FROM VOLUME IN SPARK DATAFRAME
1. DEFINE SCHEMA FOR `PAYMENTS`
2. ADD `LOAD_TIMESTAMP` AS AUDIT COLUMN

In [0]:
payments_schema = """payment_id INTEGER, customer_id INTEGER, payment_date TIMESTAMP, payment_status INTEGER,
  payment_method STRING"""

In [0]:
from pyspark.sql.functions import col, current_timestamp

payments_df = (spark.read.format("csv")
              .schema(payments_schema)
              .option("delimiter", ",")
              .load("abfss://gizmo@gizmoboxdevadls.dfs.core.windows.net/landing/external/payments")
              .withColumn("load_timestamp", current_timestamp()))
display(payments_df)
display(f'Record Count: {payments_df.count()}')

#### WRITE `PAYMENTS` INTO `DELTA FORMAT`


In [0]:
payments_df.writeTo("gizmo.bronze.payments_delta").createOrReplace()

#### VALIDATE AND QUERY THE TABLE GIZMO.BRONZE.PAYMENTS_DELTA


In [0]:
payments_external_count_df = spark.sql('''SELECT * FROM GIZMO.BRONZE.PAYMENTS_DELTA''');
print(f'Row Count: {payments_external_count_df.count()}')

In [0]:
%sql
DESC EXTENDED GIZMO.BRONZE.PAYMENTS_DELTA

In [0]:
%run /Workspace/Users/pde1409@hotmail.com/AzureDatabricks-Gizmo/AzureDatabricks-GizmoBox/01.GizmoBox/02.Config/01.config

In [0]:
try:
    verify_pipeline_counts(payments_df, payments_external_count_df, "05.IngestPaymentsExternalLocation")
except AssertionError as e:
    raise

In [0]:
from pyspark.sql import Row
from datetime import datetime
import uuid
import sys
import traceback
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType, TimestampType

# =====================================================================
# 1. INITIALIZE PIPELINE METADATA
# =====================================================================
# Capture start time at the absolute beginning of the execution
load_start_time = datetime.now()
pipeline_name = 'PySpark-05.IngestPaymentsExternalLocation'

# Establish default tracking states
status = "SUCCESS"
message = "Loaded Payments data from External Location into Bronze View"
record_count = 0

try:
    # =====================================================================
    # 2. CORE ETL LOGIC
    # =====================================================================
    
    # Step A: Extract Data using absolute path and explicit format configuration
    payments_df = (spark.read
                    .format("csv")
                    .option('header', 'true')
                    .load("abfss://gizmo@gizmoboxdevadls.dfs.core.windows.net/landing/external/payments/")) 
    
    # Step B: Execute target transformations or loading actions here
    # (Example: customers_df.write.mode("overwrite").saveAsTable("GIZMO.BRONZE.CUSTOMERS"))
    
    # Step C: Capture final evaluated source record count
    record_count = payments_df.count()
    
    # =====================================================================

except Exception as e:
    # 3. EXCEPTION HANDLING
    # If any error occurs above, catch it, flip status, and parse the trace
    status = "FAILED"
    
    exc_type, exc_value, exc_tb = sys.exc_info()
    error_details = traceback.format_exception_only(exc_type, exc_value)[0].strip()
    message = f"Pipeline failed! Error: {error_details}"
    record_count = -1  # Standard indicator flag representing an uncompleted execution

finally:
    # =====================================================================
    # 4. AUDIT & LOGGING (Guaranteed execution via finally block)
    # =====================================================================
    load_end_time = datetime.now()
    current_date = load_end_time.date()

    # Step A: Calculate Sequential Run ID for Today (Scoped to THIS specific pipeline)
    try:
        # Added F.lit() for reliable type matching and scoped the count to pipeline_name
        max_run_df = spark.table("GIZMO.AUDIT.AUDIT_LOGS") \
            .filter(
                (F.col("event_time") == F.lit(current_date)) & 
                (F.col("pipeline_name") == F.lit(pipeline_name))
            ) \
            .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
        
        max_id_row = max_run_df.collect()[0]
        next_run_int = (max_id_row["max_id"] + 1) if max_id_row["max_id"] is not None else 1
    except Exception:
        # Defaults to 1 if table is empty, uninitialized, or completely drops out
        next_run_int = 1

    # Apply 2-digit zero padding format string (e.g., 1 -> "01", 11 -> "11")
    run_id_str = f"{next_run_int:02d}"

    # Step B: Secure Notebook Cluster Context Metadata safely
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
        user_name = context.tags().apply("user")
    except Exception:
        notebook_path = "Unknown/Local"
        user_name = "System"

    # Step C: Package the metadata tracking Row
    log_entry = Row(
        log_id=str(uuid.uuid4()),
        run_id=run_id_str,                
        event_time=current_date,          
        event_type="FULL LOAD",
        source_table="text.`/Volumes/gizmo/landing/operational_data/orders/`",
        target_table="GIZMO.BRONZE.PAYMENTS_EXTERNAL",
        record_count=record_count,
        status=status,                    
        message=message,                   
        user_name=user_name,
        notebook_path=notebook_path,
        pipeline_name=pipeline_name,
        load_start_time=load_start_time,  
        load_end_time=load_end_time       
    )

    # Step D: Declare structured explicit Schema types matching Target DDL exactly
    log_schema = StructType([
        StructField("log_id", StringType(), True),
        StructField("run_id", StringType(), True),  
        StructField("event_time", DateType(), True),
        StructField("event_type", StringType(), True),
        StructField("source_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("record_count", LongType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("user_name", StringType(), True),
        StructField("notebook_path", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("load_start_time", TimestampType(), True),
        StructField("load_end_time", TimestampType(), True)
    ])

    # Step E: Instantiate log DataFrame and append transactional trace record
    log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
    log_entry_df.write.format("delta").mode("append").saveAsTable("GIZMO.AUDIT.AUDIT_LOGS")
    print(f"[AUDIT LOGGED] Status: {status} | Run ID: {run_id_str} | Count: {record_count}")

    # Step F: Force a hard stop exception for workflow orchestrators if pipeline failed
    if status == "FAILED":
        raise RuntimeError(message)

In [0]:
%python
dbutils.notebook.exit("PAYMENTS HAS BEEN LOADED SUCCESSFULLY INTO GIZMO.BRONZE.PAYMENTS_DELTA")

#### VALIDATE AUDIT RECORD COUNT GIZMO.BRONZE.AUDIT_LOGS

In [0]:
%sql
SELECT 
 run_id, 
 event_time, 
 pipeline_name, 
 record_count,
 date_format(FROM_UTC_TIMESTAMP(load_start_time, 'Asia/Kolkata'), 'yyyy-MM-dd HH:mm:ss') AS load_start_time_ist, 
  date_format(FROM_UTC_TIMESTAMP(load_end_time, 'Asia/Kolkata'), 'yyyy-MM-dd HH:mm:ss') AS load_end_time_ist
 FROM GIZMO.AUDIT.AUDIT_LOGS
 WHERE pipeline_name = 'PySpark-05.IngestPaymentsExternalLocation'
ORDER BY pipeline_name, event_time DESC, run_id ASC;